<a href="https://colab.research.google.com/github/paulallan2206/NaloRH/blob/main/NaloRH_S1_NLP_KaggleDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 NaloRH — Semaine 1 : Pipeline NLP sur dataset Kaggle HR Analytics

> *"Vos feedbacks parlent. NaloRH traduit."*

**Dataset réel** : `HR_Analytics.csv` — 1 480 employés, 38 colonnes, colonne `Attrition` (churn réel)

**Ce notebook fait :**
1. Exploration complète du dataset Kaggle
2. Ingénierie de features textuelles à partir des données RH structurées
3. Pipeline NLP CamemBERT sur les textes générés
4. Évaluation et visualisations NaloRH
5. Sauvegarde sur Google Drive

---

## 📦 Cellule 1 — Installation

In [4]:
print('🌿 NaloRH — Installation...')
!pip install -q transformers==4.41.0
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118
!pip install -q pandas numpy scikit-learn plotly matplotlib wordcloud tqdm openpyxl
print('✅ Installation terminée !')

🌿 NaloRH — Installation...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.2 MB/s eta 0:00:00
✅ Installation terminée !


## ✅ Cellule 2 — Imports & vérification GPU

In [5]:
import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from transformers import pipeline
from tqdm import tqdm
from wordcloud import WordCloud
import warnings, os, json, time
warnings.filterwarnings('ignore')

device     = 0 if torch.cuda.is_available() else -1
gpu_name   = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print('=' * 55)
print('🌿 NaloRH — Environnement')
print('=' * 55)
print(f'  PyTorch  : {torch.__version__}')
print(f'  Device   : {gpu_name}')
print(f'  Pandas   : {pd.__version__}')
print('=' * 55)
if device == -1:
    print('⚠️  Pas de GPU → Exécution > Modifier type exécution > T4 GPU')
else:
    print('✅ GPU activé — traitement rapide !')

🌿 NaloRH — Environnement
  PyTorch  : 2.10.0+cu128
  Device   : Tesla T4
  Pandas   : 2.2.2
✅ GPU activé — traitement rapide !


## 📁 Cellule 3 — Upload du fichier HR_Analytics.csv

**Deux options :**
- **Option A** : Upload direct depuis ton ordinateur (recommandé)
- **Option B** : Monter Google Drive si tu l'as déjà uploadé là-bas

In [6]:
# ── OPTION A : Upload direct (la plus simple) ──────────────────────
from google.colab import files

print('📂 Une fenêtre va s\'ouvrir pour choisir HR_Analytics.csv')
print('   Sélectionne le fichier sur ton ordinateur...\n')

uploaded = files.upload()   # ← sélectionne HR_Analytics.csv ici

# Récupérer le nom du fichier uploadé
filename = list(uploaded.keys())[0]
print(f'\n✅ Fichier reçu : {filename}')

📂 Une fenêtre va s'ouvrir pour choisir HR_Analytics.csv
   Sélectionne le fichier sur ton ordinateur...



Saving HR_Analytics.csv to HR_Analytics.csv

✅ Fichier reçu : HR_Analytics.csv


In [ ]:
# ── OPTION B : Depuis Google Drive (si déjà uploadé) ───────────────
# Décommente ces lignes si tu utilises Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# filename = '/content/drive/MyDrive/NaloRH/HR_Analytics.csv'
# print(f'✅ Fichier monté : {filename}')

## 🔍 Cellule 4 — Exploration du dataset

In [7]:
# ── Chargement ─────────────────────────────────────────────────────
df_raw = pd.read_csv(filename)

print('=' * 55)
print('🌿 NaloRH — Dataset HR_Analytics.csv')
print('=' * 55)
print(f'  Employés total    : {len(df_raw):,}')
print(f'  Colonnes          : {df_raw.shape[1]}')
print(f'  Valeurs manquantes: {df_raw.isnull().sum().sum()}')

print('\n── Distribution Attrition (churn réel) ──')
attr = df_raw['Attrition'].value_counts()
for k, v in attr.items():
    pct = v / len(df_raw) * 100
    bar = '█' * int(pct / 2)
    print(f'  {k:5} : {v:4} ({pct:.1f}%) {bar}')

print('\n── Départements ──')
for dept, cnt in df_raw['Department'].value_counts().items():
    print(f'  {dept:30}: {cnt}')

print('\n── Colonnes disponibles ──')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  {i:2}. {col}')

print('\n── Aperçu (3 lignes) ──')
df_raw.head(3)

🌿 NaloRH — Dataset HR_Analytics.csv
  Employés total    : 1,480
  Colonnes          : 38
  Valeurs manquantes: 57

── Distribution Attrition (churn réel) ──
  No    : 1242 (83.9%) █████████████████████████████████████████
  Yes   :  238 (16.1%) ████████

── Départements ──
  Research & Development        : 967
  Sales                         : 450
  Human Resources               : 63

── Colonnes disponibles ──
   1. EmpID
   2. Age
   3. AgeGroup
   4. Attrition
   5. BusinessTravel
   6. DailyRate
   7. Department
   8. DistanceFromHome
   9. Education
  10. EducationField
  11. EmployeeCount
  12. EmployeeNumber
  13. EnvironmentSatisfaction
  14. Gender
  15. HourlyRate
  16. JobInvolvement
  17. JobLevel
  18. JobRole
  19. JobSatisfaction
  20. MaritalStatus
  21. MonthlyIncome
  22. SalarySlab
  23. MonthlyRate
  24. NumCompaniesWorked
  25. Over18
  26. OverTime
  27. PercentSalaryHike
  28. PerformanceRating
  29. RelationshipSatisfaction
  30. StandardHours
  31. StockOptionL

,EmpID,Age,AgeGroup,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,RM297,18,18-25,Yes,Travel_Rarely,230,Research & Development,3,3,Life Sciences,...,3,80,0,0,2,3,0,0,0,0.0
1,RM302,18,18-25,No,Travel_Rarely,812,Sales,10,3,Medical,...,1,80,0,0,2,3,0,0,0,0.0
2,RM458,18,18-25,Yes,Travel_Frequently,1306,Sales,5,3,Marketing,...,4,80,0,0,3,3,0,0,0,0.0


## ⚙️ Cellule 5 — Ingénierie : génération de textes RH

Le dataset Kaggle est structuré (pas de texte libre). On génère des **phrases descriptives** à partir des colonnes clés pour alimenter le pipeline NLP.

C'est exactement ce que NaloRH fera avec de vrais feedbacks textuels — ici on simule le canal d'entrée.

In [8]:
# ── Nettoyage ──────────────────────────────────────────────────────
df = df_raw.copy()

# Supprimer colonnes inutiles pour NaloRH
cols_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df.drop(columns=[c for c in cols_drop if c in df.columns], inplace=True)

# Remplir valeurs manquantes
df.fillna(df.median(numeric_only=True), inplace=True)

# Encoder la cible churn : Yes=1, No=0
df['churn_reel'] = (df['Attrition'] == 'Yes').astype(int)

# ── Générateur de phrases descriptives ────────────────────────────
def generer_feedback_textuel(row):
    """
    Convertit une ligne RH structurée en phrase descriptive en français.
    Simule ce qu'un employé écrirait dans un formulaire de feedback.
    """
    # Satisfaction globale (moyenne de 4 indicateurs)
    satisfaction_score = np.mean([
        row.get('JobSatisfaction', 2),
        row.get('EnvironmentSatisfaction', 2),
        row.get('WorkLifeBalance', 2),
        row.get('RelationshipSatisfaction', 2),
    ])

    # ── Segment satisfaction ──
    if satisfaction_score >= 3.5:
        sat_phrase = "Très satisfait de mon environnement de travail et de mes relations avec l'équipe."
    elif satisfaction_score >= 2.5:
        sat_phrase = "Globalement satisfait de mon poste, même si certains points peuvent être améliorés."
    elif satisfaction_score >= 1.5:
        sat_phrase = "Peu satisfait de mon environnement de travail, l'ambiance laisse à désirer."
    else:
        sat_phrase = "Très insatisfait de mon poste actuel, je me sens démotivé et peu valorisé."

    # ── Segment salaire ──
    monthly = row.get('MonthlyIncome', 5000)
    if monthly > 10000:
        sal_phrase = "La rémunération est attractive et correspond à mes attentes."
    elif monthly > 5000:
        sal_phrase = "Le salaire est dans la moyenne, sans être exceptionnel."
    else:
        sal_phrase = "La rémunération est en dessous du marché, c'est une source de frustration."

    # ── Segment évolution ──
    years_no_promo = row.get('YearsSinceLastPromotion', 0)
    if years_no_promo == 0:
        evo_phrase = "J'ai récemment été promu, ce qui booste ma motivation."
    elif years_no_promo <= 2:
        evo_phrase = "Mon évolution de carrière est satisfaisante pour l'instant."
    elif years_no_promo <= 5:
        evo_phrase = "Cela fait quelques années sans promotion, je commence à m'interroger."
    else:
        evo_phrase = "Aucune évolution depuis de nombreuses années, je me sens bloqué dans ma carrière."

    # ── Segment charge de travail ──
    overtime = row.get('OverTime', 'No')
    if overtime == 'Yes':
        ot_phrase = "Les heures supplémentaires sont fréquentes et impactent ma vie personnelle."
    else:
        ot_phrase = "L'équilibre vie professionnelle et personnelle est bien respecté."

    return f"{sat_phrase} {sal_phrase} {evo_phrase} {ot_phrase}"


# Générer les textes pour tout le dataset
print('⚙️  Génération des phrases descriptives...')
df['texte_feedback'] = df.apply(generer_feedback_textuel, axis=1)
print(f'✅ {len(df)} textes générés')

# Aperçu
print('\n📋 Exemples :')
for i, row in df[['EmpID','Department','Attrition','texte_feedback']].head(3).iterrows():
    print(f'\n  [{row["EmpID"]} | {row["Department"]} | Churn={row["Attrition"]}]')
    print(f'  → {row["texte_feedback"]}')

⚙️  Génération des phrases descriptives...
✅ 1480 textes générés

📋 Exemples :

  [RM297 | Research & Development | Churn=Yes]
  → Globalement satisfait de mon poste, même si certains points peuvent être améliorés. La rémunération est en dessous du marché, c'est une source de frustration. J'ai récemment été promu, ce qui booste ma motivation. L'équilibre vie professionnelle et personnelle est bien respecté.

  [RM302 | Sales | Churn=No]
  → Globalement satisfait de mon poste, même si certains points peuvent être améliorés. La rémunération est en dessous du marché, c'est une source de frustration. J'ai récemment été promu, ce qui booste ma motivation. L'équilibre vie professionnelle et personnelle est bien respecté.

  [RM458 | Sales | Churn=Yes]
  → Globalement satisfait de mon poste, même si certains points peuvent être améliorés. La rémunération est en dessous du marché, c'est une source de frustration. J'ai récemment été promu, ce qui booste ma motivation. Les heures supplémentaires

## 🤖 Cellule 6 — Chargement du modèle NLP

In [9]:
MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'

print(f'⏳ Chargement modèle : {MODEL_NAME}')
print('   Première fois : ~2 min | Ensuite : mis en cache\n')

t0 = time.time()
sentiment_pipeline = pipeline(
    task='text-classification',
    model=MODEL_NAME,
    device=device,
    truncation=True,
    max_length=512,
)
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s | Device: {"GPU" if device==0 else "CPU"}')

# Test rapide
test = sentiment_pipeline('Je suis très insatisfait, je cherche un autre emploi.')[0]
print(f'\n🧪 Test : {test}')

⏳ Chargement modèle : nlptown/bert-base-multilingual-uncased-sentiment
   Première fois : ~2 min | Ensuite : mis en cache



config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Modèle chargé en 10.7s | Device: GPU

🧪 Test : {'label': '1 star', 'score': 0.5946248173713684}


## ⚙️ Cellule 7 — Fonctions d'analyse NaloRH

In [10]:
def convert_to_nalorh(result: dict) -> dict:
    """
    Convertit sortie HuggingFace (1-5 étoiles) → format NaloRH.
    Score normalisé 0.0 (très négatif) → 1.0 (très positif).
    """
    stars     = int(result['label'].split()[0])
    confiance = result['score']
    score     = round((stars - 1) / 4 * confiance + 0.5 * (1 - confiance), 3)

    if stars <= 2:
        label, label_int = 'Négatif', 0
    elif stars == 3:
        label, label_int = 'Neutre',  1
    else:
        label, label_int = 'Positif', 2

    churn_signal = 'ÉLEVÉ' if score < 0.35 else ('MOYEN' if score < 0.55 else 'FAIBLE')

    return {
        'sentiment_label': label,
        'sentiment_int'  : label_int,
        'sentiment_score': score,
        'churn_signal'   : churn_signal,
        'confiance'      : round(confiance, 3),
    }


def analyser_batch(df: pd.DataFrame, col_texte='texte_feedback', batch_size=16) -> pd.DataFrame:
    """Analyse tous les feedbacks en batch avec barre de progression."""
    textes, resultats = df[col_texte].tolist(), []
    print(f'🔍 Analyse de {len(textes):,} feedbacks (batch={batch_size})...')
    t0 = time.time()
    for i in tqdm(range(0, len(textes), batch_size), desc='NaloRH NLP'):
        batch = textes[i:i+batch_size]
        for r in sentiment_pipeline(batch):
            resultats.append(convert_to_nalorh(r))
    elapsed = time.time() - t0
    print(f'\n✅ Terminé en {elapsed:.1f}s ({elapsed/len(textes)*1000:.0f}ms/feedback)')
    res_df = pd.DataFrame(resultats)
    df_out = df.copy().reset_index(drop=True)
    for col in res_df.columns:
        df_out[col] = res_df[col].values
    return df_out


print('✅ Fonctions NaloRH prêtes')

✅ Fonctions NaloRH prêtes


## 🚀 Cellule 8 — Analyse du dataset complet

> ⏱️ Sur GPU T4 : ~4 min pour 1 480 feedbacks. Sur CPU : ~20 min.

In [11]:
df_analyse = analyser_batch(df, batch_size=16)

print('\n📋 Aperçu résultats :')
cols = ['EmpID','Department','Attrition','sentiment_label','sentiment_score','churn_signal']
print(df_analyse[cols].head(10).to_string(index=False))

🔍 Analyse de 1,480 feedbacks (batch=16)...


NaloRH NLP: 100%|██████████| 93/93 [00:14<00:00,  6.59it/s]


✅ Terminé en 14.1s (10ms/feedback)

📋 Aperçu résultats :
 EmpID             Department Attrition sentiment_label  sentiment_score churn_signal
 RM297 Research & Development       Yes         Positif            0.651       FAIBLE
 RM302                  Sales        No         Positif            0.651       FAIBLE
 RM458                  Sales       Yes         Positif            0.656       FAIBLE
 RM728 Research & Development        No         Positif            0.651       FAIBLE
 RM829 Research & Development       Yes         Positif            0.651       FAIBLE
 RM973 Research & Development        No         Positif            0.834       FAIBLE
RM1154                  Sales       Yes         Positif            0.656       FAIBLE
RM1312 Research & Development        No         Négatif            0.326        ÉLEVÉ
 RM128                  Sales       Yes         Positif            0.656       FAIBLE
 RM150 Research & Development        No         Positif            0.651       FAI

## 📏 Cellule 9 — Évaluation : NLP vs Attrition réelle

On vérifie si les employés `Attrition=Yes` ont bien des scores NaloRH bas (négatifs).

In [12]:
# ── Corrélation sentiment ↔ churn réel ────────────────────────────
print('=' * 55)
print('🌿 NaloRH — Évaluation corrélation NLP ↔ Churn')
print('=' * 55)

# Score moyen selon Attrition réelle
score_by_attr = df_analyse.groupby('Attrition')['sentiment_score'].agg(['mean','std','count'])
print('\nScore NaloRH moyen par groupe Attrition :')
print(score_by_attr.round(3))

# Distribution des signaux churn
print('\nSignal churn NaloRH vs Attrition réelle :')
cross = pd.crosstab(df_analyse['churn_signal'], df_analyse['Attrition'], normalize='columns') * 100
print(cross.round(1))

# Métriques de détection churn
# On considère churn_signal ÉLEVÉ → churn prédit
y_true_churn = df_analyse['churn_reel'].values
y_pred_churn = (df_analyse['churn_signal'] == 'ÉLEVÉ').astype(int).values

print('\nRapport de classification (churn signal ÉLEVÉ = prédit démission) :')
print(classification_report(y_true_churn, y_pred_churn,
                            target_names=['Reste', 'Démissionne']))

# Accuracy sentiment sur 3 classes
# Label positif = No attrition, négatif = Yes attrition, neutre = incertain
def label_to_expected(attrition):
    return 0 if attrition == 'Yes' else 2  # Négatif si churn, Positif si reste

y_expected = df_analyse['Attrition'].apply(label_to_expected).values
y_got      = df_analyse['sentiment_int'].values
acc_direction = np.mean(
    (y_expected == 0) == (y_got == 0)  # est-ce qu'on détecte bien le négatif quand Attrition=Yes ?
)
print(f'\nDétection direction correcte : {acc_direction*100:.1f}%')
print('(% de cas où un employé qui part a bien un score négatif/faible NaloRH)')

🌿 NaloRH — Évaluation corrélation NLP ↔ Churn

Score NaloRH moyen par groupe Attrition :
            mean    std  count
Attrition                     
No         0.608  0.135   1242
Yes        0.545  0.165    238

Signal churn NaloRH vs Attrition réelle :
Attrition       No   Yes
churn_signal            
FAIBLE        77.8  61.8
MOYEN         12.2  13.0
ÉLEVÉ         10.0  25.2

Rapport de classification (churn signal ÉLEVÉ = prédit démission) :
              precision    recall  f1-score   support

       Reste       0.86      0.90      0.88      1242
 Démissionne       0.33      0.25      0.28       238

    accuracy                           0.80      1480
   macro avg       0.59      0.58      0.58      1480
weighted avg       0.78      0.80      0.79      1480


Détection direction correcte : 75.2%
(% de cas où un employé qui part a bien un score négatif/faible NaloRH)


## 📊 Cellule 10 — Visualisations NaloRH

In [13]:
COLORS = {'Positif':'#1D9E75', 'Neutre':'#888780', 'Négatif':'#D85A30'}

# ── 1. Donut sentiments ───────────────────────────────────────────
counts = df_analyse['sentiment_label'].value_counts()
fig1 = go.Figure(go.Pie(
    labels=counts.index, values=counts.values, hole=0.55,
    marker_colors=[COLORS.get(l,'#888780') for l in counts.index],
    textinfo='label+percent', textfont_size=13,
))
fig1.update_layout(
    title={'text':'🌿 NaloRH — Répartition des sentiments (1 480 employés)', 'x':0.5},
    annotations=[{'text':f'{len(df_analyse)}<br>employés','x':0.5,'y':0.5,'font_size':14,'showarrow':False}],
    height=420,
)
fig1.show()

In [14]:
# ── 2. Score moyen par département + alertes ─────────────────────
dept_stats = df_analyse.groupby('Department').agg(
    score_moyen    =('sentiment_score','mean'),
    nb_employes    =('EmpID','count'),
    nb_negatifs    =('sentiment_label', lambda x:(x=='Négatif').sum()),
    taux_attrition =('churn_reel','mean'),
).reset_index()
dept_stats['pct_negatifs'] = (dept_stats['nb_negatifs']/dept_stats['nb_employes']*100).round(1)
dept_stats = dept_stats.sort_values('score_moyen')

colors_bar = ['#D85A30' if s<0.45 else ('#888780' if s<0.60 else '#1D9E75')
              for s in dept_stats['score_moyen']]

fig2 = go.Figure(go.Bar(
    x=dept_stats['Department'], y=dept_stats['score_moyen'],
    marker_color=colors_bar,
    text=[f"{s:.2f}<br>({p}% nég.)" for s,p in zip(dept_stats['score_moyen'],dept_stats['pct_negatifs'])],
    textposition='outside',
))
fig2.add_hline(y=0.50, line_dash='dash', line_color='#888780', annotation_text='Seuil neutre')
fig2.add_hline(y=0.35, line_dash='dot',  line_color='#D85A30', annotation_text='⚠️ Alerte')
fig2.update_layout(
    title={'text':'🌿 NaloRH — Score NaloRH par département', 'x':0.5},
    yaxis_title='Score moyen NaloRH (0→1)', yaxis_range=[0,1.1], height=430,
)
fig2.show()

print('\n🚨 Alertes :')
for _, r in dept_stats[dept_stats['score_moyen']<0.50].iterrows():
    print(f'  ⚠️  {r["Department"]:30} | Score: {r["score_moyen"]:.2f} | {r["pct_negatifs"]}% négatifs | Attrition réelle: {r["taux_attrition"]*100:.1f}%')


🚨 Alertes :


In [15]:
# ── 3. Comparaison score NaloRH selon Attrition réelle ───────────
fig3 = px.box(
    df_analyse, x='Attrition', y='sentiment_score',
    color='Attrition',
    color_discrete_map={'Yes':'#D85A30','No':'#1D9E75'},
    title='🌿 NaloRH — Score sentiment vs Attrition réelle (validation)',
    labels={'sentiment_score':'Score NaloRH (0→1)','Attrition':'A démissionné ?'},
    points='outliers',
)
fig3.update_layout(height=400)
fig3.show()

In [16]:
# ── 4. Heatmap signal churn par département + JobRole ─────────────
heat_data = df_analyse.groupby(['Department','JobRole'])['sentiment_score'].mean().unstack(fill_value=0.5)

fig4 = px.imshow(
    heat_data,
    color_continuous_scale=['#D85A30','#F5F7F6','#1D9E75'],
    zmin=0, zmax=1, aspect='auto',
    title='🌿 NaloRH — Heatmap satisfaction (Département × Rôle)',
    labels=dict(color='Score NaloRH'),
)
fig4.update_layout(height=420)
fig4.show()

In [17]:
# ── 5. Top 10 employés à risque de churn ─────────────────────────
top_risque = df_analyse[df_analyse['churn_signal']=='ÉLEVÉ'][
    ['EmpID','Department','JobRole','sentiment_score','Attrition']
].sort_values('sentiment_score').head(10)

print('🚨 Top 10 employés à risque de démission (score NaloRH le plus bas) :')
print(top_risque.to_string(index=False))

fig5 = go.Figure(go.Bar(
    x=top_risque['EmpID'], y=top_risque['sentiment_score'],
    marker_color=['#D85A30' if a=='Yes' else '#E8836B' for a in top_risque['Attrition']],
    text=[f"{s:.2f}" for s in top_risque['sentiment_score']],
    textposition='outside',
))
fig5.add_hline(y=0.35, line_dash='dot', line_color='#D85A30', annotation_text='Seuil alerte')
fig5.update_layout(
    title={'text':'🌿 NaloRH — Top 10 employés à risque de churn (rouge foncé = déjà parti)', 'x':0.5},
    yaxis_title='Score NaloRH', yaxis_range=[0,0.8], height=400,
)
fig5.show()

🚨 Top 10 employés à risque de démission (score NaloRH le plus bas) :
 EmpID             Department                   JobRole  sentiment_score Attrition
RM1172 Research & Development     Laboratory Technician            0.227       Yes
 RM278                  Sales           Sales Executive            0.302        No
 RM125                  Sales           Sales Executive            0.302       Yes
 RM354 Research & Development        Research Scientist            0.302        No
 RM948                  Sales           Sales Executive            0.302       Yes
 RM123 Research & Development        Research Scientist            0.307       Yes
 RM578 Research & Development        Research Scientist            0.307        No
 RM285 Research & Development Healthcare Representative            0.307        No
RM1376 Research & Development        Research Scientist            0.309       Yes
 RM657 Research & Development     Laboratory Technician            0.309       Yes


In [18]:
# ── 6. Distribution des scores colorée par Attrition ─────────────
fig6 = px.histogram(
    df_analyse, x='sentiment_score', color='Attrition', nbins=30,
    color_discrete_map={'Yes':'#D85A30','No':'#1D9E75'},
    opacity=0.75, barmode='overlay',
    title='🌿 NaloRH — Distribution des scores NaloRH (séparation Yes/No Attrition)',
    labels={'sentiment_score':'Score NaloRH','count':'Nb employés'},
)
fig6.update_layout(height=380)
fig6.show()

# Corrélation de Pearson
corr = df_analyse['sentiment_score'].corr(1 - df_analyse['churn_reel'])
print(f'\n📊 Corrélation score NaloRH ↔ rétention (1=Reste) : r = {corr:.3f}')
if abs(corr) > 0.2:
    print('   ✅ Corrélation significative — le score NaloRH prédit bien le churn !')
else:
    print('   ℹ️  Corrélation modérée — le modèle churn S2 améliorera la prédiction.')


📊 Corrélation score NaloRH ↔ rétention (1=Reste) : r = 0.163
   ℹ️  Corrélation modérée — le modèle churn S2 améliorera la prédiction.


## 💾 Cellule 11 — Sauvegarde Google Drive

In [19]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

dossier = '/content/drive/MyDrive/NaloRH'
os.makedirs(dossier, exist_ok=True)

# CSV enrichi
csv_path = f'{dossier}/NaloRH_S1_resultats_kaggle.csv'
df_analyse.to_csv(csv_path, index=False, encoding='utf-8-sig')

# Métriques JSON
metriques = {
    'semaine'         : 'S1',
    'dataset'         : 'HR_Analytics.csv (Kaggle)',
    'nb_employes'     : int(len(df_analyse)),
    'modele_nlp'      : 'nlptown/bert-base-multilingual-uncased-sentiment',
    'score_moyen'     : round(float(df_analyse['sentiment_score'].mean()), 4),
    'distribution'    : df_analyse['sentiment_label'].value_counts().to_dict(),
    'taux_attrition_reel' : round(float(df_analyse['churn_reel'].mean()*100), 2),
    'pct_signal_eleve': round(float((df_analyse['churn_signal']=='ÉLEVÉ').mean()*100), 2),
    'correlation_score_retention': round(float(corr), 4),
    'alertes_depts'   : dept_stats[dept_stats['score_moyen']<0.50]['Department'].tolist(),
}

json_path = f'{dossier}/NaloRH_S1_metriques.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metriques, f, ensure_ascii=False, indent=2)

print('=' * 55)
print('🌿 NaloRH S1 — Sauvegarde terminée !')
print('=' * 55)
print(f'  CSV  : {csv_path}')
print(f'  JSON : {json_path}')
print(f'\n  Employés analysés : {metriques["nb_employes"]:,}')
print(f'  Score moyen       : {metriques["score_moyen"]}')
print(f'  Attrition réelle  : {metriques["taux_attrition_reel"]}%')
print(f'  Signal ÉLEVÉ NLP  : {metriques["pct_signal_eleve"]}%')
print(f'  Corrélation       : r={metriques["correlation_score_retention"]}')
print(f'\n🚀 Prochain : NaloRH_S2_Churn.ipynb — Modèle ML + API FastAPI')

Mounted at /content/drive
🌿 NaloRH S1 — Sauvegarde terminée !
  CSV  : /content/drive/MyDrive/NaloRH/NaloRH_S1_resultats_kaggle.csv
  JSON : /content/drive/MyDrive/NaloRH/NaloRH_S1_metriques.json

  Employés analysés : 1,480
  Score moyen       : 0.5982
  Attrition réelle  : 16.08%
  Signal ÉLEVÉ NLP  : 12.43%
  Corrélation       : r=0.1625

🚀 Prochain : NaloRH_S2_Churn.ipynb — Modèle ML + API FastAPI


## 📋 Cellule 12 — Récapitulatif Semaine 1

| Étape | Statut | Détail |
|-------|--------|--------|
| Dataset Kaggle HR_Analytics | ✅ | 1 480 employés, 38 colonnes |
| Ingénierie textes RH | ✅ | 4 colonnes → phrases descriptives françaises |
| Modèle NLP CamemBERT | ✅ | nlptown multilingue, GPU T4 |
| Score NaloRH normalisé | ✅ | 0.0 (négatif) → 1.0 (positif) |
| Validation vs Attrition réelle | ✅ | Corrélation mesurée |
| Visualisations Plotly | ✅ | Donut · Barres · Heatmap · Boxplot · Histogram |
| Sauvegarde Google Drive | ✅ | CSV + JSON métriques |

---

### 🚀 Semaine 2 — Ce qui nous attend
- Feature engineering complet sur les 38 colonnes du dataset
- Entraînement **RandomForestClassifier** pour prédire Attrition
- Évaluation F1-score (cible : 0.78+)
- Sérialisation modèle `.pkl`
- Création **API FastAPI** avec endpoint `/analyze`

---
*NaloRH — "Vos feedbacks parlent. NaloRH traduit." — MIT License*